# Gradient maps visualization

### Load model and titles

In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import random

st_model = SentenceTransformer('BAAI/bge-large-en-v1.5')
base_model = st_model._first_module().auto_model
tokenizer = st_model.tokenizer

DATASET = 'Amazon_Sports_and_Outdoors'

ITEM_PATH = f'../../{DATASET}/{DATASET}.item'

df = pd.read_csv(ITEM_PATH, delimiter='\t')
df = df[['item_id:token', 'title:token']]

titles = df['title:token'].astype('str').tolist()

title = random.choice(titles)

2025-10-24 00:53:43.880128: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-24 00:53:43.918741: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-24 00:53:45.153059: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Convert title to tokens

In [2]:
inputs = tokenizer(title, return_tensors='pt', add_special_tokens=True)

device = next(base_model.parameters()).device
input_ids = inputs['input_ids'].to(device)
attention_mask = inputs['attention_mask'].to(device)

embeds = base_model.embeddings.word_embeddings(input_ids)
embeds.retain_grad()

### Calculate vanilla saliency map

In [3]:
output = base_model(inputs_embeds=embeds, attention_mask=attention_mask)
hidden_state = output.last_hidden_state

sentence_embedding = hidden_state.mean(dim=1)

target = sentence_embedding.norm()
target.backward()

token_gradients = embeds.grad.norm(dim=-1).squeeze().detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze())
token_importance = [(t, w.item()) for t, w in zip(tokens, token_gradients) if t not in tokenizer.all_special_tokens]

print(f'Original title: {title}')
for token, weight in token_importance:
    print(f'{token}: {weight:.2f}')

Original title: Galco Ankle Lite / Ankle Holster for Ruger LCP, KelTec P3AT, P32
gal: 1.31
##co: 0.68
ankle: 0.96
lit: 0.83
##e: 0.33
/: 0.16
ankle: 0.84
holster: 1.54
for: 0.23
rug: 1.35
##er: 0.46
lc: 1.18
##p: 0.46
,: 0.16
ke: 0.52
##lt: 0.69
##ec: 0.51
p: 0.29
##3: 0.61
##at: 0.53
,: 0.19
p: 0.29
##32: 0.79
